In [ ]:
!pip install datasets

In [2]:
import pandas as pd
from datasets import Dataset

# Đọc file
df = pd.read_csv("/teamspace/studios/this_studio/.lightning_studio/data.csv")  # Hoặc load từ dataframe đã có
df = df.dropna(subset=["src", "alt"])  # Xóa dòng thiếu thông tin

# Tạo dataset HF
dataset = Dataset.from_pandas(df[["src", "alt"]])

In [ ]:
!pip install transformers accelerate

In [5]:
from transformers import AutoProcessor, AutoModelForCausalLM

processor = AutoProcessor.from_pretrained("microsoft/git-base")
model = AutoModelForCausalLM.from_pretrained("microsoft/git-base")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
from PIL import Image
import requests
from io import BytesIO

def preprocess(example):
    from PIL import Image
    import requests
    from io import BytesIO

    try:
        response = requests.get(example["src"], timeout=5)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        # Nếu ảnh lỗi, trả về None để loại bỏ
        return {"pixel_values": None, "input_ids": None, "attention_mask": None, "labels": None}

    # Tiền xử lý
    inputs = processor(images=image, text=example["alt"], padding="max_length", truncation=True, return_tensors="pt")
    inputs = {k: v.squeeze() for k, v in inputs.items()}
    inputs["labels"] = inputs["input_ids"].clone()
    return inputs

In [ ]:
dataset = dataset.map(preprocess)

In [ ]:
# Bỏ những ảnh lỗi (labels là None)
dataset = dataset.filter(lambda example: example["labels"] is not None)

In [ ]:
dataset.save_to_disk("filtered_dataset")


In [1]:
from datasets import load_from_disk
dataset = load_from_disk("filtered_dataset")


Loading dataset from disk:   0%|          | 0/59 [00:00<?, ?it/s]

In [ ]:
!pip install -U transformers


In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
from transformers.training_args_seq2seq import Seq2SeqTrainingArguments
help(Seq2SeqTrainingArguments)


In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.51.3


In [3]:
from datasets import load_from_disk

# # Load lại dataset
# dataset = load_from_disk("filtered_dataset")

# Dùng phương thức train_test_split của HuggingFace, nhanh và tối ưu bộ nhớ hơn
split_dataset = dataset.train_test_split(test_size=0.2)

# Lấy ra train và eval
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# In size
print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")


Train size: 37964
Eval size: 9491


In [6]:
from transformers import Trainer, TrainingArguments
import torch

# Định nghĩa hàm để thực hiện đánh giá
def eval_steps_callback(model, eval_dataset, step_interval=500):
    if step % step_interval == 0:
        results = model.evaluate(eval_dataset)
        print(f"Evaluation at step {step}: {results}")

# Cấu hình TrainingArguments
training_args = TrainingArguments(
    output_dir="./git-finetuned-food",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
)

# Định nghĩa Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Huấn luyện mô hình và theo dõi đánh giá
for step in range(training_args.num_train_epochs):
    trainer.train()
    eval_steps_callback(model, eval_dataset, step_interval=500)


Step,Training Loss
100,4.908200
200,0.301100
300,0.277200
400,0.238500
500,0.239700
600,0.218100
700,0.243000
800,0.222300
900,0.222000
1000,0.211000


AttributeError: 'GitForCausalLM' object has no attribute 'evaluate'

In [1]:
from transformers import GitProcessor, GitForCausalLM

processor = GitProcessor.from_pretrained("microsoft/git-base")
model = GitForCausalLM.from_pretrained("./git-finetuned-food/checkpoint-28473")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [23]:
from PIL import Image

# Load ảnh
image = Image.open("/teamspace/studios/this_studio/.lightning_studio/32ada647-6c0e-45a4-bb83-cd1442d6897e.jpg")

# Tiền xử lý ảnh
inputs = processor(images=image, return_tensors="pt")

# Sinh caption (KHÔNG truyền thẳng pixel_values)
generated_ids = model.generate(
    **inputs,
    max_new_tokens=100,              # Không cần quá dài
    do_sample=False,                # KHÔNG lấy mẫu, chọn xác suất cao nhất
    num_beams=5,                    # Beam search giúp tăng độ chính xác
    early_stopping=True,           # Dừng sớm nếu caption đã hợp lý
    repetition_penalty=1.3         # Giảm lặp từ nhẹ
)

# Decode caption
caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Caption:", caption)

Caption: this recipe is by chef and takes.


In [20]:
model.eval()  # <- Câu thần chú của anh trai mỗi khi caption bị xàm


GitForCausalLM(
  (git): GitModel(
    (embeddings): GitEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(1024, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (image_encoder): GitVisionModel(
      (vision_model): GitVisionTransformer(
        (embeddings): GitVisionEmbeddings(
          (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
          (position_embedding): Embedding(197, 768)
        )
        (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (encoder): GitVisionEncoder(
          (layers): ModuleList(
            (0-11): 12 x GitVisionEncoderLayer(
              (self_attn): GitVisionAttention(
                (k_proj): Linear(in_features=768, out_features=768, bias=True)
                (v_proj): Linear(in_features=768, out_features=768, bias=True)
             

In [21]:
caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
caption = caption.strip().capitalize()
print("Clean caption:", caption)


Clean caption: Waffles with syrup and cinnamon on a white background.


In [22]:
from transformers import GitProcessor, GitForCausalLM

model = GitForCausalLM.from_pretrained("microsoft/git-base")
processor = GitProcessor.from_pretrained("microsoft/git-base")

inputs = processor(images=image, return_tensors="pt")
model.eval()
generated_ids = model.generate(**inputs, max_new_tokens=100)
caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Original caption:", caption)


Original caption: waffles with syrup and a syrup.
